### Independent weight validation
Re-derives stratum, cluster, household, and respondent weights from the raw inputs using formulas re-typed from the methodology (does **not** call `calculate_three_stage_weights()`), then cross-checks every stage against `data/2_cleaned/02_fem_survey_benin_weighted.csv`. A mismatch here means either `src/funcs.py` or this notebook has a bug -- they were written independently, so agreement is a real check, not a tautology.

In [ ]:
import pandas as pd
import geopandas as gpd
from src.config import *

TOL = 1e-6

In [ ]:
elig_rename = {
    'COL5A60A6EACEB1FDF1': 'geom_id', 'COL5A60A6EACE8FA0D5': 'place',
    'COL5A60A6EACEE635C8': 'province', 'COL5A60A6EACEEB6B90': 'population',
    'COL5A60A6EACE23D663': 'proportion', 'COL5A61F9C2A9DB5103': 'strata_description',
    'COL5A61F9C2A950E5C1': 'n_clusters_target', 'COL5A60A6EACE9C5044': 'longitude',
    'COL5A60A6EACE9C9991': 'latitude',
}
e1 = gpd.read_file(DIR_META / "benin sample eligible 1.gpkg").rename(columns=elig_rename)
e2 = gpd.read_file(DIR_META / "benin sample eligible 2.gpkg").rename(columns=elig_rename)
# File 3: 5 south-region clusters (Doga; Agonlin-Houégbo, Zassa; Hlanhonou; Tohouè; Gnanli,
# Kpanroun, Près de Zalli, Ayokpo) that were surveyed as replacements but were missing from
# files 1/2 -- added 2026-08-19, confirmed to match the surveyed file's geom_id/province.
e3 = gpd.read_file(DIR_META / "benin sample eligible 3.gpkg").rename(columns=elig_rename)
eligible = pd.concat([e1, e2, e3], ignore_index=True).drop_duplicates(subset="geom_id", keep="first")

chosen_rename = {
    '1FFFE3D41E414BF4F43F': 'geom_id', '233E4AB11C414BF4F43F': 'place',
    '32B28DB955414BF4F43F': 'province', '4E0E32C8B9414BF4F43F': 'population',
    '5929598760414BF4F43F': 'proportion', '6350D2B3BE414BF4F43F': 'strata_description',
    '76CC79C927414BF4F43F': 'n_clusters_target', '83A1D65C64414BF4F43F': 'longitude',
    '935C6905CD414BF4F43F': 'latitude',
}
chosen = gpd.read_file(DIR_META / "Benin Sampling Surveyed.gpkg")
chosen = chosen.drop_duplicates(subset="Name", keep="first")
chosen["Name"] = chosen["Name"].replace("NA", "Unnamed Cluster (NA)")
chosen = chosen.rename(columns=chosen_rename)

# Population linked on geom_id from the original sample-draw spreadsheets (2026-08-19),
# same source src/... 02b_calculate_weights_KH.ipynb now uses, rather than the gpkg's own
# hashed population column.
sample_draw_v2 = pd.read_csv(DIR_META / "benin_sample_(n=60)-2026-02-11 - sample_draw_v2.csv", header=2)
sample_draw_v21 = pd.read_csv(DIR_META / "benin_sample_(n=60)-2026-02-11 - sample_draw_v2.1.csv", header=2)
population_by_geom_id = pd.concat(
    [sample_draw_v2[["geom_id", "population"]], sample_draw_v21[["geom_id", "population"]]],
    ignore_index=True
).drop_duplicates(subset="geom_id", keep="first").set_index("geom_id")["population"]
chosen["population"] = chosen["geom_id"].map(population_by_geom_id).astype(float)

d = pd.read_csv(DIR_WEIGHTED_DATA)
avg_household_size = 5.5
results = {}

#### Stage 1a -- stratum weight
For province $h$: $\quad w_h = \dfrac{N_h / N}{k_h / K}$, where $N_h$ = eligible population in stratum $h$ (from the eligible frame), $N$ = total eligible population, $k_h$ = clusters actually visited in stratum $h$, $K$ = total clusters visited. This corrects for unequal allocation across strata -- a stratum that got proportionally *more* clusters than its population share gets a *smaller* weight per cluster, and vice versa.

In [ ]:
strata_counts = chosen.groupby("province").size()                 # k_h
stratum_pops = eligible.groupby("province")["population"].sum()   # N_h
total_pop = stratum_pops.sum()                                     # N
total_clusters = len(chosen)                                       # K

stratum_weight = {}
for prov in chosen["province"].unique():
    if prov not in stratum_pops.index:
        continue  # province not in the eligible frame at all
    k_h = strata_counts[prov]
    N_h = stratum_pops[prov]
    stratum_weight[prov] = (N_h / total_pop) / (k_h / total_clusters)

check = d["province"].map(stratum_weight)
diff = (check - d["stratum_weights"]).abs()
results["stratum_weights"] = (diff.max(skipna=True), (diff > TOL).sum())
pd.Series(stratum_weight).sort_values()

#### Stage 1b -- within-stratum cluster (PPS) weight
Within stratum $h$, each cluster $i$'s share of the *eligible frame's* population proportion is renormalized to sum to 1 within the stratum: $p_i = \text{proportion}_i / \sum_{\text{stratum } h} \text{proportion}$. With $k_h$ draws taken from the stratum, the (sampling-without-replacement-approximated) inclusion probability is $\pi_i = 1-(1-p_i)^{k_h}$, and the cluster weight is $1/\pi_i$ -- clusters with a larger population share had a higher chance of selection, so they get downweighted relative to small clusters that got lucky.

In [ ]:
chosen["cluster_weight"] = pd.NA
for prov in chosen["province"].unique():
    mask = chosen["province"] == prov
    if prov not in stratum_pops.index:
        continue
    k_h = strata_counts[prov]
    elig_prov = eligible[eligible["province"] == prov]
    prop_sum = elig_prov["proportion"].sum()
    prop_within = (elig_prov.set_index("geom_id")["proportion"] / prop_sum).to_dict()
    p_i = chosen.loc[mask, "geom_id"].map(prop_within)
    inclusion_prob = 1 - (1 - p_i) ** k_h
    chosen.loc[mask, "cluster_weight"] = 1.0 / inclusion_prob

cw_map = chosen.set_index("Name")["cluster_weight"]
check = d["location"].map(cw_map).astype(float)
diff = (check - d["cluster_weight"]).abs()
results["cluster_weight"] = (diff.max(skipna=True), (diff > TOL).sum())

s1_map = chosen.set_index("Name")["cluster_weight"].astype(float) * chosen["province"].map(stratum_weight).set_axis(chosen["Name"])
check = d["location"].map(s1_map)
diff = (check - d["stage1_weight"]).abs()
results["stage1_weight"] = (diff.max(skipna=True), (diff > TOL).sum())

print(f"clusters with no match in the eligible frame (NaN cluster_weight): "
      f"{chosen['cluster_weight'].isna().sum()} / {len(chosen)}")
chosen.loc[chosen['cluster_weight'].isna(), ['Name', 'province']]

#### Stage 2 -- household weight (approximation, non-probability)
$\text{stage2}_i = \dfrac{\text{cluster population}_i / \text{avg household size}}{\text{households actually visited in cluster } i}$. This is an approximation because households weren't randomly sampled within cluster -- it just downweights clusters where, relative to their estimated total household count, an unusually large share of households happened to get visited.

In [ ]:
households_visited = d.groupby("location")["household_id"].nunique()
total_hh_est = chosen.set_index("Name")["population"] / avg_household_size
s2_map = total_hh_est / households_visited
check = d["location"].map(s2_map)
diff = (check - d["stage2_weight"]).abs()
results["stage2_weight"] = (diff.max(skipna=True), (diff > TOL).sum())

#### Stage 3 -- respondent weight
$\text{stage3} = $ number of eligible adults listed in the household. One respondent was randomly drawn from that many candidates, so they represent all of them: $\text{stage3}$ = inverse of a $1/n$ selection probability = $n$.

In [ ]:
diff = (d["eligible_adults_in_hh"] - d["stage3_weight"]).abs()
results["stage3_weight"] = (diff.max(skipna=True), (diff > TOL).sum())

#### Combine, non-response adjust, normalize
$\text{combined} = \text{stage1} \times \text{stage2} \times \text{stage3}$.

$\text{response rate} = \text{random\_draw\_attempts} / \text{eligible\_adults\_in\_hh}$; $\text{combined\_adjusted} = \text{combined} / \text{response rate}$.

$\text{normalized} = \text{combined\_adjusted} \times n / \sum \text{combined\_adjusted}$, so the weights sum to the achieved sample size $n$ (mean weight = 1).

In [ ]:
combined = d["stage1_weight"] * d["stage2_weight"] * d["stage3_weight"]
diff = (combined - d["combined_weight"]).abs()
results["combined_weight"] = (diff.max(skipna=True), (diff > TOL).sum())

# Re-derive household_response_rate independently, including the same three adjustments
# src/funcs.py applies (capped 2026-08-19, Borgou-day-1 and Plateau-language pooled-rate
# overrides confirmed by the user 2026-08-19) -- see the comments in funcs.py's
# NON-RESPONSE ADJUSTMENT section for the full rationale on each.
response_rate = (d["random_draw_attempts"] / d["eligible_adults_in_hh"]).clip(upper=1.0)

start_date = pd.to_datetime(d["starttime"], format="mixed", dayfirst=True).dt.date
borgou_day1 = (d["province"] == "Borgou") & (start_date == pd.Timestamp("2026-07-16").date())
if borgou_day1.any():
    borgou_rest = (d["province"] == "Borgou") & (~borgou_day1)
    response_rate.loc[borgou_day1] = response_rate.loc[borgou_rest].mean()

PLATEAU_LANGUAGE_NONRESPONSE_N = 8
plateau_mask = d["province"] == "Plateau"
if plateau_mask.any():
    n_completed_plateau = plateau_mask.sum()
    plateau_rate = n_completed_plateau / (n_completed_plateau + PLATEAU_LANGUAGE_NONRESPONSE_N)
    response_rate.loc[plateau_mask] *= plateau_rate

diff = (response_rate - d["household_response_rate"]).abs()
results["household_response_rate"] = (diff.max(skipna=True), (diff > TOL).sum())

combined_adj = combined / response_rate
diff = (combined_adj - d["combined_weight_adjusted"]).abs()
results["combined_weight_adjusted"] = (diff.max(skipna=True), (diff > TOL).sum())

n = len(d)
normalized = combined_adj * n / combined_adj.sum()
diff = (normalized - d["normalized_weight"]).abs()
results["normalized_weight"] = (diff.max(skipna=True), (diff > TOL).sum())

In [ ]:
print(f"{'component':<26} {'max |diff|':>14} {'# mismatches':>14}   status")
print("-" * 70)
all_ok = True
for name, (maxdiff, n_bad) in results.items():
    ok = (n_bad == 0) or pd.isna(maxdiff)
    all_ok &= ok
    status = "OK" if ok else "MISMATCH"
    print(f"{name:<26} {maxdiff:>14.6g} {n_bad:>14}   {status}")

print()
print(f"Rows with NaN stage1_weight (cluster missing from the eligible frame, excluded above): "
      f"{d['stage1_weight'].isna().sum()} / {len(d)}")
print()
print("ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED -- see MISMATCH rows above")